In [ ]:
# CELL 1: Connection and fixed top-50 temporal selection experiment
# Paste private sf_options = {...} here. No credentials are supplied.
# Runtime: Spark Snowflake connector, numpy, pandas, sklearn, torch, matplotlib.
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
if "sf_options" not in globals() or not isinstance(sf_options, dict) or "spark" not in globals():
    raise RuntimeError("Supply sf_options on the approved Spark runtime.")
sf_options_dl_poc = dict(sf_options)
sf_options_dl_poc.update(sfDatabase="DSVC_TAKEDA_TA_PRIVATE", sfSchema="DS_ML")
PREFIX = "TAK861_TX_READY_V63_DL_POC"
EXPERIMENT_PREFIX = PREFIX + "_TEMPORAL_SELECTION_V1"
REFERENCE_RUN_ID = "RUN_001"
RUN_ID = "P001"  # New ID for changed ranking/training settings or code; same ID in all four notebooks.
PLAN_SEED = 42
import re
if not re.fullmatch(r"[A-Z][A-Z0-9_]{0,15}", RUN_ID):
    raise ValueError("RUN_ID must be a short uppercase identifier.")
PREPARATION_TABLE = EXPERIMENT_PREFIX + "_PREPARATION"
SPLIT_AUDIT_TABLE = EXPERIMENT_PREFIX + "_SPLIT_AUDIT"
REFERENCE_MODEL_TABLE = PREFIX + "_MODEL_" + REFERENCE_RUN_ID
RUN_PREFIX = EXPERIMENT_PREFIX + "_" + RUN_ID
INTERNAL_TABLE = RUN_PREFIX + "_INTERNAL"
RANK_MODEL_TABLE = RUN_PREFIX + "_RANK_MODEL"
SELECTION_TABLE = RUN_PREFIX + "_FEATURE_SELECTION"
MODEL_TABLE = RUN_PREFIX + "_MODEL"
EVALUATION_TABLE = RUN_PREFIX + "_EVALUATION"

TOP_K = 50
PERMUTATION_REPEATS = 5
PERMUTATION_SEED = 1042
FEATURES_PER_CHUNK = 32  # Completed chunks are saved for restart; does not change ranking.
MODEL_SETTINGS = dict(seq_len=12, d_model=128, n_heads=4, encoder_layers=2,
                      feedforward_dim=256, dropout=.2)
TRAINING_SETTINGS = dict(seed=42, epochs=20, patience=5, min_delta=1e-4, batch_size=64,
                        learning_rate=.001, weight_decay=.0001, grad_clip=1., device="auto")
# Preserve original AP checkpoint selection and training settings.
# Importance uses lift drop; AP drop breaks ranking ties. TEST is never scored here.


In [ ]:
# CELL 2: Embedded original temporal encoder, training, permutation and storage helpers
"""Embedded by the delivery notebooks; no repository dependency at runtime."""
import base64
import hashlib
import io
import json
import math
import re
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd


def canonical_json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)


def digest_json(value):
    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()


def read_sf(suffix):
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", f"{PREFIX}_{suffix}").load())


def table_exists(table):
    if not re.fullmatch(r"[A-Z][A-Z0-9_]*", table):
        raise ValueError("Use uppercase letters, numbers and underscores in table names.")
    query = ("SELECT TABLE_NAME FROM DSVC_TAKEDA_TA_PRIVATE.INFORMATION_SCHEMA.TABLES "
             f"WHERE TABLE_SCHEMA = 'DS_ML' AND TABLE_NAME = '{table}'")
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("query", query).load().limit(1).count() > 0)


def checked_metadata(frame, include_split=False):
    columns = ["PATIENT_ID", "END_DT", "RESP"]
    if include_split:
        columns += ["SPLIT", "SPLIT_CONFIG"]
    out = frame.loc[:, columns].copy()
    if out.empty or out.isna().any().any():
        raise ValueError("Snapshot metadata must be nonempty and contain no nulls.")
    if not out.PATIENT_ID.map(lambda x: isinstance(x, str) and bool(x)).all():
        raise ValueError("PATIENT_ID must retain its original nonempty string value.")
    dates = pd.to_datetime(out.END_DT, errors="raise")
    if dates.dt.tz is not None or not dates.eq(dates.dt.normalize()).all():
        raise ValueError("END_DT must be a date without an intraday time/timezone.")
    out["END_DT"] = dates.dt.strftime("%Y-%m-%d")
    if not out.RESP.isin([0, 1]).all():
        raise ValueError("RESP must be exactly 0 or 1 before conversion.")
    out["RESP"] = out.RESP.astype("int64")
    if out.duplicated(["PATIENT_ID", "END_DT"]).any():
        raise ValueError("Duplicate patient/date keys in snapshot metadata.")
    if include_split:
        if set(out.SPLIT) != {"train", "validation", "test"}:
            raise ValueError("Expected the saved train, validation and test assignments.")
        if out.groupby("PATIENT_ID", observed=True).SPLIT.nunique().gt(1).any():
            raise ValueError("Patient overlap between splits.")
        if len(out.SPLIT_CONFIG.unique()) != 1:
            raise ValueError("The split contains inconsistent creation settings.")
        for _, part in out.groupby("SPLIT", observed=True):
            if set(part.RESP) != {0, 1}:
                raise ValueError("Each split must contain both response classes.")
    return out.sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)


def validate_metadata_pair(snapshot_frame, manifest_frame, features):
    source = checked_metadata(snapshot_frame)
    manifest = checked_metadata(manifest_frame, include_split=True)
    if not source.equals(manifest[["PATIENT_ID", "END_DT", "RESP"]]):
        raise ValueError("Frozen split and source snapshots differ in keys or labels.")
    creation = json.loads(manifest.SPLIT_CONFIG.iloc[0])
    # Match the feature-order hash created in the completed split notebook.
    feature_order_hash = hashlib.sha256(
        json.dumps(features, ensure_ascii=False).encode("utf-8")).hexdigest()
    if creation.get("feature_order_sha256") != feature_order_hash or creation.get("n_timesteps") != 12:
        raise ValueError("Feature order or timesteps differ from the frozen split.")
    return manifest, creation


def new_tensor_buffer(metadata, feature_count, seq_len=12):
    temporary = tempfile.TemporaryDirectory(prefix="dl_poc_")
    path = Path(temporary.name) / "counts.float32"
    X = np.memmap(path, dtype="<f4", mode="w+", shape=(len(metadata), seq_len, feature_count))
    return temporary, X


def fill_tensor(X, metadata, sequence_rows):
    """Place rows by canonical keys, independent of Spark partition order."""
    positions = {(r.PATIENT_ID, r.END_DT): i for i, r in enumerate(metadata.itertuples())}
    seen = np.zeros(len(metadata), dtype=bool)
    seq_len, feature_count = X.shape[1:]
    for row in sequence_rows:
        raw_date = row["END_DT"]
        if raw_date is None or row["PATIENT_ID"] is None:
            raise ValueError("Null monthly snapshot key.")
        date = pd.Timestamp(raw_date)
        if date.tzinfo is not None or date != date.normalize():
            raise ValueError("Monthly END_DT is not an exact date.")
        key = (row["PATIENT_ID"], date.strftime("%Y-%m-%d"))
        if key not in positions:
            raise ValueError("Unexpected monthly snapshot key.")
        i = positions[key]
        if seen[i] or row["RESP"] != int(metadata.RESP.iloc[i]):
            raise ValueError("Duplicate monthly snapshot or changed label.")
        sequence = row["SEQUENCE"]
        steps = [month["T"] for month in sequence]
        if len(sequence) != seq_len or any(t is None for t in steps):
            raise ValueError("Expected exactly 12 complete timesteps per snapshot.")
        # Check original values before any integer conversion.
        if sorted(steps) != list(range(seq_len)):
            raise ValueError("Timesteps must be unique integers 0 through 11.")
        sequence = sorted(sequence, key=lambda month: month["T"])
        values = np.asarray([month["V"] for month in sequence], dtype=np.float32)
        if values.shape != (seq_len, feature_count):
            raise ValueError("Monthly feature shape does not match the vocabulary.")
        if not np.isfinite(values).all() or (values < 0).any():
            raise ValueError("Counts must be finite, nonnegative float32 values with no nulls.")
        X[i] = values
        seen[i] = True
    if not seen.all():
        raise ValueError("Monthly data is missing original snapshots, including zero-activity sequences.")
    X.flush()


def input_fingerprints(X, metadata, features):
    tensor_hash = hashlib.sha256()
    tensor_hash.update(canonical_json(list(X.shape)).encode("utf-8"))
    for start in range(0, len(X), 128):
        tensor_hash.update(np.asarray(X[start:start + 128], dtype="<f4").tobytes(order="C"))
    records = [[str(r.PATIENT_ID), str(r.END_DT), int(r.RESP), str(r.SPLIT)]
               for r in metadata.itertuples()]
    return {"model_input_sha256": tensor_hash.hexdigest(),
            "snapshot_manifest_sha256": digest_json(records),
            "feature_names_sha256": digest_json(features),
            "split_config_sha256": digest_json(json.loads(metadata.SPLIT_CONFIG.iloc[0]))}


def load_inputs():
    from pyspark.sql import functions as F
    mapping = read_sf("FEATURE_MAP").orderBy("FEATURE_INDEX").collect()
    if not mapping or [r["FEATURE_INDEX"] for r in mapping] != list(range(len(mapping))):
        raise ValueError("Invalid feature indices.")
    features = [r["FEATURE_NAME"] for r in mapping]
    aliases = [r["FEATURE_COLUMN"] for r in mapping]
    if (aliases != [f"F{i:04d}" for i in range(len(features))]
            or not all(isinstance(f, str) and f for f in features)
            or len(set(features)) != len(features)):
        raise ValueError("Invalid feature names, aliases or order.")
    source = read_sf("SNAPSHOTS").select("PATIENT_ID", "END_DT", "RESP").toPandas()
    frozen = read_sf("PATIENT_SPLIT").select(
        "PATIENT_ID", "END_DT", "RESP", "SPLIT", "SPLIT_CONFIG").toPandas()
    metadata, creation = validate_metadata_pair(source, frozen, features)
    observed = (len(metadata), metadata.PATIENT_ID.nunique(), int(metadata.RESP.sum()), len(features))
    if observed != (23151, 12447, 1345, 1028):
        raise ValueError(f"V63 population changed: snapshots/patients/positives/features = {observed}.")
    monthly = read_sf("TENSOR_MONTHLY")
    if set(monthly.columns) != set(["PATIENT_ID", "END_DT", "RESP", "TIME_STEP"] + aliases):
        raise ValueError("Monthly table columns differ from the frozen feature map.")
    print("Building the model input in a temporary driver file (about 1.06 GiB).", flush=True)
    grouped = (monthly.groupBy("PATIENT_ID", "END_DT", "RESP")
        .agg(F.collect_list(F.struct(
            F.col("TIME_STEP").alias("T"),
            F.array(*[F.when(F.col(name) >= 0, F.col(name).cast("float"))
                      .otherwise(F.lit(None).cast("float")) for name in aliases]).alias("V")
        )).alias("SEQUENCE"))
        .repartition(128))
    temporary, X = new_tensor_buffer(metadata, len(features))
    try:
        fill_tensor(X, metadata, grouped.toLocalIterator(prefetchPartitions=False))
        hashes = input_fingerprints(X, metadata, features)
        X.flags.writeable = False
    except BaseException:
        X._mmap.close()
        temporary.cleanup()
        raise
    y = metadata.RESP.to_numpy(dtype=np.float32)
    indices = {name: np.flatnonzero(metadata.SPLIT.to_numpy() == name)
               for name in ("train", "validation", "test")}
    print(f"Validated tensor shape {X.shape}; patient overlap = 0.", flush=True)
    return {"X": X, "y": y, "metadata": metadata, "features": features,
            "indices": indices, "hashes": hashes, "split_creation": creation,
            "temporary_directory": temporary}


def pack_artifacts(artifacts, chunk_size=50000):
    rows = []
    for name, blob in artifacts.items():
        encoded = base64.b64encode(blob).decode("ascii")
        pieces = [encoded[i:i + chunk_size] for i in range(0, len(encoded), chunk_size)] or [""]
        digest = hashlib.sha256(blob).hexdigest()
        rows.extend((name, i, len(pieces), len(blob), digest, piece)
                    for i, piece in enumerate(pieces))
    return rows


def unpack_artifacts(rows, expected_names):
    groups = {}
    for row in rows:
        name, i, count, size, digest, payload = tuple(row)
        if any(value != int(value) for value in (i, count, size)):
            raise ValueError("Nonintegral artifact chunk metadata.")
        groups.setdefault(name, []).append((int(i), int(count), int(size), digest, payload))
    if set(groups) != set(expected_names):
        raise ValueError("Missing or unexpected saved artifacts.")
    result = {}
    for name, pieces in groups.items():
        pieces.sort(key=lambda p: p[0])
        count, size, digest = pieces[0][1:4]
        if (count < 1 or size < 0 or len(pieces) != count
                or [p[0] for p in pieces] != list(range(count))
                or any(p[1:4] != (count, size, digest) for p in pieces)):
            raise ValueError("Missing, duplicate or inconsistent artifact chunks.")
        blob = base64.b64decode("".join(p[4] for p in pieces), validate=True)
        if len(blob) != size or hashlib.sha256(blob).hexdigest() != digest:
            raise ValueError("Artifact length/hash mismatch.")
        result[name] = blob
    return result


ARTIFACT_COLUMNS = ["ARTIFACT_NAME", "CHUNK_INDEX", "CHUNK_COUNT", "BYTE_LENGTH", "SHA256", "PAYLOAD_BASE64"]


def read_artifacts(table, expected_names):
    rows = (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load().select(*ARTIFACT_COLUMNS).collect())
    return unpack_artifacts(rows, expected_names)


def save_artifacts(table, artifacts):
    # A matching existing result can be verified after an interrupted read-back.
    if table_exists(table):
        if read_artifacts(table, artifacts) != artifacts:
            raise FileExistsError("Destination contains different artifacts; choose a new RUN_ID.")
        print(f"Existing artifacts verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")
        return
    schema = ("ARTIFACT_NAME STRING, CHUNK_INDEX INT, CHUNK_COUNT INT, "
              "BYTE_LENGTH LONG, SHA256 STRING, PAYLOAD_BASE64 STRING")
    frame = spark.createDataFrame(pack_artifacts(artifacts), schema=schema)
    (frame.write.format("snowflake").options(**sf_options_dl_poc)
     .option("dbtable", table).option("truncate_columns", "off")
     .mode("errorifexists").save())
    if read_artifacts(table, artifacts) != artifacts:
        raise ValueError("Saved artifact read-back differs from the completed run.")
    print(f"Saved and verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")

REFERENCE_TRAINING_ARTIFACT_NAMES = {
    "checkpoint.pt", "training_summary.json", "training_history.csv", "training_history.png"
}


def bind_report_to_reference(report, artifacts, reference_run_id, stage):
    """Bind a source audit to the original completed run without loading PyTorch."""
    if (not isinstance(reference_run_id, str)
            or not re.fullmatch(r"[A-Z][A-Z0-9_]*", reference_run_id)):
        raise ValueError("REFERENCE_RUN_ID must be the original saved uppercase run identifier.")
    if not isinstance(artifacts, dict) or set(artifacts) != REFERENCE_TRAINING_ARTIFACT_NAMES:
        raise ValueError("The original reference run must contain its exact four training artifacts.")
    try:
        summary = json.loads(artifacts["training_summary.json"])
    except (TypeError, ValueError, UnicodeError) as error:
        raise ValueError("The original reference training summary is not valid JSON.") from error
    if (not isinstance(summary, dict) or summary.get("training_complete") is not True
            or summary.get("run_id") != reference_run_id):
        raise ValueError("The reference training run is incomplete or belongs to another RUN_ID.")
    hashes = summary.get("input_hashes")
    expected_hashes = {"model_input_sha256", "snapshot_manifest_sha256",
                       "feature_names_sha256", "split_config_sha256"}
    if (not isinstance(hashes, dict) or set(hashes) != expected_hashes
            or any(not isinstance(value, str) or not re.fullmatch(r"[0-9a-f]{64}", value)
                   for value in hashes.values())):
        raise ValueError("The original reference summary lacks valid full input fingerprints.")
    checkpoint = artifacts["checkpoint.pt"]
    if not isinstance(checkpoint, bytes) or not checkpoint:
        raise ValueError("The original reference checkpoint is missing or empty.")
    stages = {
        "preparation": ("reuse_and_validate_saved_v63_tensor",
                        ("model_input_sha256", "feature_names_sha256")),
        "split": ("reuse_original_patient_split",
                  ("snapshot_manifest_sha256", "split_config_sha256", "feature_names_sha256")),
    }
    if stage not in stages or not isinstance(report, dict) or report.get("mode") != stages[stage][0]:
        raise ValueError("Unexpected source audit stage for reference verification.")
    for field in stages[stage][1]:
        if report.get(field) != hashes[field]:
            raise ValueError(
                f"Current saved inputs differ from original {reference_run_id} on {field}. "
                "Restore the original saved inputs; matching population counts alone are insufficient.")
    bound = dict(report)
    bound["reference_run_id"] = reference_run_id
    bound["reference_checkpoint_sha256"] = hashlib.sha256(checkpoint).hexdigest()
    bound["reference_input_hashes"] = dict(hashes)
    bound["checks"] = dict(report.get("checks", {}), matches_original_reference_run=True)
    return bound


def validate_feature_mapping(mapping):
    if not mapping or [r["FEATURE_INDEX"] for r in mapping] != list(range(len(mapping))):
        raise ValueError("Invalid feature indices in the saved FEATURE_MAP.")
    features = [r["FEATURE_NAME"] for r in mapping]
    aliases = [r["FEATURE_COLUMN"] for r in mapping]
    if (aliases != [f"F{i:04d}" for i in range(len(features))]
            or not all(isinstance(f, str) and f for f in features)
            or len(set(features)) != len(features)):
        raise ValueError("Invalid saved feature names, aliases or order.")
    return features, aliases


def original_snapshot_hash(metadata):
    records = [[str(r.PATIENT_ID), str(r.END_DT), int(r.RESP)]
               for r in metadata.itertuples()]
    return digest_json(records)


def validate_original_population(metadata, features):
    observed = (len(metadata), int(metadata.PATIENT_ID.nunique()),
                int(metadata.RESP.sum()), len(features))
    if observed != (23151, 12447, 1345, 1028):
        raise ValueError(
            "This handoff reuses the completed V63 cohort; "
            f"snapshots/patients/positives/features changed to {observed}.")


def make_preparation_report(X, metadata, features, source_prefix):
    # fill_tensor has already validated all rows, labels, timesteps and counts.
    validate_original_population(metadata, features)
    if tuple(X.shape) != (len(metadata), 12, len(features)):
        raise ValueError("The saved model-input shape must be snapshots x 12 x features.")
    tensor_hash = hashlib.sha256()
    tensor_hash.update(canonical_json(list(X.shape)).encode("utf-8"))
    zero_months = 0
    zero_snapshots = 0
    for start in range(0, len(X), 128):
        block = np.asarray(X[start:start + 128], dtype="<f4")
        tensor_hash.update(block.tobytes(order="C"))
        empty_months = np.all(block == 0, axis=2)
        zero_months += int(empty_months.sum())
        zero_snapshots += int(np.all(empty_months, axis=1).sum())
    groups = {}
    for feature in features:
        group = feature.split("__", 1)[0] if "__" in feature else "OTHER"
        groups[group] = groups.get(group, 0) + 1
    return {
        "report_version": 1,
        "mode": "reuse_and_validate_saved_v63_tensor",
        "source_prefix": source_prefix,
        "source_tables": [source_prefix + "_" + suffix
                          for suffix in ("FEATURE_MAP", "SNAPSHOTS", "TENSOR_MONTHLY")],
        "n_snapshots": len(metadata),
        "n_patients": int(metadata.PATIENT_ID.nunique()),
        "n_positive_snapshots": int(metadata.RESP.sum()),
        "n_negative_snapshots": int(len(metadata) - metadata.RESP.sum()),
        "n_features": len(features),
        "n_timesteps": 12,
        "n_monthly_rows": int(X.shape[0] * X.shape[1]),
        "tensor_shape": list(X.shape),
        "model_input_dtype": "little_endian_float32",
        "minimum_snapshot_end_date": str(metadata.END_DT.min()),
        "maximum_snapshot_end_date": str(metadata.END_DT.max()),
        "zero_activity_months": zero_months,
        "zero_activity_snapshots": zero_snapshots,
        "feature_group_counts": groups,
        "model_input_sha256": tensor_hash.hexdigest(),
        "source_snapshots_sha256": original_snapshot_hash(metadata),
        "feature_names_sha256": digest_json(features),
        "checks": {
            "exact_saved_population": True,
            "unique_patient_date_keys": True,
            "exact_binary_labels": True,
            "exact_12_unique_timesteps": True,
            "complete_snapshot_coverage": True,
            "finite_nonnegative_counts": True,
            "zero_activity_sequences_preserved": True,
        },
        "scope": "Saved tensor validation only; raw claims and mappings were not regenerated.",
        "feature_selection": "Deferred to TRAIN-only processing in notebook 03.",
    }


def make_split_audit_report(metadata, creation, features, source_prefix):
    # validate_metadata_pair has already checked source coverage and all split rules.
    validate_original_population(metadata, features)
    expected = {
        "train": (16256, 8712, 941),
        "validation": (3481, 1867, 202),
        "test": (3414, 1868, 202),
    }
    summary = []
    patient_sets = {}
    for split_name in ("train", "validation", "test"):
        part = metadata.loc[metadata.SPLIT == split_name]
        observed = (len(part), int(part.PATIENT_ID.nunique()), int(part.RESP.sum()))
        if observed != expected[split_name]:
            raise ValueError(
                f"Saved {split_name} split differs from RUN_001: {observed}. "
                "Restore the original manifest; this notebook does not reshuffle patients.")
        patient_sets[split_name] = set(part.PATIENT_ID)
        summary.append({
            "split": split_name,
            "snapshots": observed[0],
            "patients": observed[1],
            "positive_snapshots": observed[2],
            "negative_snapshots": observed[0] - observed[2],
            "positive_fraction": observed[2] / observed[0],
            "minimum_snapshot_end_date": str(part.END_DT.min()),
            "maximum_snapshot_end_date": str(part.END_DT.max()),
        })
    overlaps = {
        "train_validation": len(patient_sets["train"] & patient_sets["validation"]),
        "train_test": len(patient_sets["train"] & patient_sets["test"]),
        "validation_test": len(patient_sets["validation"] & patient_sets["test"]),
    }
    if any(overlaps.values()):
        raise ValueError("A patient appears in more than one split.")
    records = [[str(r.PATIENT_ID), str(r.END_DT), int(r.RESP), str(r.SPLIT)]
               for r in metadata.itertuples()]
    return {
        "report_version": 1,
        "mode": "reuse_original_patient_split",
        "source_prefix": source_prefix,
        "source_table": source_prefix + "_PATIENT_SPLIT",
        "split_summary": summary,
        "patient_overlap_counts": overlaps,
        "source_snapshots_sha256": original_snapshot_hash(metadata),
        "snapshot_manifest_sha256": digest_json(records),
        "feature_names_sha256": digest_json(features),
        "split_config_sha256": digest_json(creation),
        "saved_split_creation": creation,
        "new_assignments_created": False,
        "checks": {
            "source_keys_and_labels_match": True,
            "both_classes_in_each_split": True,
            "patient_overlap_zero": True,
            "original_split_counts_match": True,
            "feature_order_and_timesteps_match": True,
        },
    }


def compare_preparation_to_split(preparation_report, split_report):
    for report in (preparation_report, split_report):
        if (not report.get("reference_run_id") or not report.get("reference_checkpoint_sha256")
                or not isinstance(report.get("reference_input_hashes"), dict)):
            raise ValueError("Both source audits must be bound to the original reference training run.")
    for field in ("source_prefix", "source_snapshots_sha256", "feature_names_sha256",
                  "reference_run_id", "reference_checkpoint_sha256", "reference_input_hashes"):
        if preparation_report.get(field) != split_report.get(field):
            raise ValueError(f"Stage 01 preparation and frozen split disagree on {field}.")
    if preparation_report.get("mode") != "reuse_and_validate_saved_v63_tensor":
        raise ValueError("Unexpected preparation report; run notebook 01 from this delivery.")
    return True
"""Small sequence classifier; zero-activity months remain real timesteps."""

from dataclasses import dataclass

import torch
from torch import nn


@dataclass(frozen=True)
class ModelConfig:
    input_dim: int
    seq_len: int = 12
    d_model: int = 128
    n_heads: int = 4
    encoder_layers: int = 2
    feedforward_dim: int = 256
    dropout: float = 0.2

    def __post_init__(self):
        sizes = (self.input_dim, self.seq_len, self.d_model, self.n_heads,
                 self.encoder_layers, self.feedforward_dim)
        if any(not isinstance(n, int) or isinstance(n, bool) or n <= 0 for n in sizes):
            raise ValueError("All model dimensions must be positive integers.")
        if self.d_model % self.n_heads:
            raise ValueError("d_model must be divisible by n_heads.")
        if not 0 <= self.dropout < 1:
            raise ValueError("dropout must be in [0, 1).")


class ClaimsTransformer(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        self.projection = nn.Linear(config.input_dim, config.d_model)
        self.position = nn.Parameter(torch.empty(1, config.seq_len, config.d_model))
        nn.init.normal_(self.position, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=config.d_model, nhead=config.n_heads,
            dim_feedforward=config.feedforward_dim, dropout=config.dropout,
            activation="relu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            layer, num_layers=config.encoder_layers, enable_nested_tensor=False,
        )
        # TransformerEncoder clones the initial layer; initialize matrix weights
        # independently so the layers do not begin with identical weights.
        for encoder_layer in self.encoder.layers:
            for parameter in encoder_layer.parameters():
                if parameter.dim() > 1:
                    nn.init.xavier_uniform_(parameter)
        self.norm = nn.LayerNorm(config.d_model)
        self.head = nn.Sequential(
            nn.Linear(config.d_model, 64), nn.ReLU(),
            nn.Dropout(config.dropout), nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        expected = (self.config.seq_len, self.config.input_dim)
        if x.ndim != 3 or tuple(x.shape[1:]) != expected:
            raise ValueError(f"Expected [batch, {expected[0]}, {expected[1]}] input.")
        if not x.is_floating_point():
            raise TypeError("Transformer input must be floating point.")
        # A zero month is observed absence of activity, not padding. A causal
        # mask is unnecessary because every included month precedes the cutoff.
        hidden = self.projection(x) + self.position
        pooled = self.norm(self.encoder(hidden)).mean(dim=1)
        return self.head(pooled).squeeze(-1)

"""Classification metrics and a threshold selected only on VALIDATION."""

import numpy as np
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)


def _validate(y, probabilities):
    labels = np.asarray(y)
    raw_scores = np.asarray(probabilities)
    if np.iscomplexobj(raw_scores) or (
        raw_scores.dtype == object
        and any(isinstance(value, (complex, np.complexfloating)) for value in raw_scores.flat)
    ):
        raise ValueError("Probabilities must be real values, not complex numbers.")
    scores = np.asarray(raw_scores, dtype=float)
    if labels.ndim != 1 or scores.ndim != 1 or len(labels) != len(scores) or not len(labels):
        raise ValueError("Labels and probabilities must be aligned, nonempty 1-D arrays.")
    if not np.isin(labels, [0, 1]).all():
        raise ValueError("Labels must be binary 0/1.")
    if not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
        raise ValueError("Probabilities must be finite and in [0, 1].")
    return labels.astype(np.int64), scores


def select_validation_threshold(y, probabilities) -> float:
    """Maximize VALIDATION F1; an exact tie uses the highest threshold.

    The caller must provide VALIDATION labels and scores, never TEST. Predictions
    are positive when score >= threshold. Equal scores are never split, and
    integer cross-products identify exact F1 ties without rounding ambiguity.
    """
    labels, scores = _validate(y, probabilities)
    if len(np.unique(labels)) != 2:
        raise ValueError("Threshold selection requires both VALIDATION classes.")
    order = np.argsort(scores, kind="stable")[::-1]
    ranked_scores = scores[order]
    true_positives = np.cumsum(labels[order], dtype=np.int64)
    group_ends = np.r_[np.flatnonzero(ranked_scores[:-1] != ranked_scores[1:]), len(labels) - 1]
    total_positives = int(labels.sum())
    best_numerator, best_denominator = 0, 1
    best_threshold = float(ranked_scores[0])
    for end in group_ends:
        # F1 = 2 TP / (number selected + total positives). Python integers
        # keep cross-products exact and avoid fixed-width integer overflow.
        numerator = 2 * int(true_positives[end])
        denominator = int(end) + 1 + total_positives
        if numerator * best_denominator > best_numerator * denominator:
            best_numerator, best_denominator = numerator, denominator
            best_threshold = float(ranked_scores[end])
        # Descending thresholds retain the highest cutoff on an exact tie.
    return best_threshold


def classification_metrics(y, probabilities, threshold: float) -> dict:
    labels, scores = _validate(y, probabilities)
    if not np.isfinite(threshold) or not 0 <= threshold <= 1:
        raise ValueError("The fixed classification threshold must be in [0, 1].")
    predicted = (scores >= threshold).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=[0, 1]).ravel()
    return {
        "average_precision": float(average_precision_score(labels, scores)) if labels.sum() else None,
        "roc_auc": float(roc_auc_score(labels, scores)) if len(np.unique(labels)) == 2 else None,
        "precision": float(precision_score(labels, predicted, zero_division=0)),
        "recall": float(recall_score(labels, predicted, zero_division=0)),
        "f1": float(f1_score(labels, predicted, zero_division=0)),
        "threshold": float(threshold),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

"""Seed configuration and aggregate-only runtime provenance."""

import os
import platform
import random

import numpy as np
import sklearn
import torch


def seed_everything(seed: int) -> None:
    if not isinstance(seed, int) or isinstance(seed, bool) or not 0 <= seed < 2**32:
        raise ValueError("seed must be an integer in [0, 2**32).")
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def seed_worker(worker_id: int) -> None:
    """Use the DataLoader's seeded generator for each worker process."""
    del worker_id
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def resolve_device(device: str = "auto") -> torch.device:
    if device == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    resolved = torch.device(device)
    if resolved.type not in {"cpu", "cuda"}:
        raise ValueError("Supported devices are auto, cpu, or cuda[:index].")
    if resolved.type == "cuda" and not torch.cuda.is_available():
        raise ValueError("CUDA requested but unavailable; set device='cpu'.")
    return resolved


def runtime_metadata() -> dict:
    return {
        "python_version": platform.python_version(),
        "numpy_version": str(np.__version__),
        "sklearn_version": str(sklearn.__version__),
        "torch_version": str(torch.__version__),
        "cuda_version": str(torch.version.cuda),
        "deterministic_algorithms_requested": torch.are_deterministic_algorithms_enabled(),
        "determinism_scope": "Best effort within a fixed device and software environment; unsupported operations warn.",
    }

"""Notebook-local training; consumes the verified memory-mapped tensor."""
from dataclasses import asdict
from datetime import datetime, timezone

from torch.utils.data import Dataset, DataLoader


class SequenceDataset(Dataset):
    def __init__(self, data, split):
        self.data = data
        self.indices = data["indices"][split]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        row = int(self.indices[index])
        x = torch.from_numpy(np.array(self.data["X"][row], dtype=np.float32, copy=True))
        return x, torch.tensor(float(self.data["y"][row]), dtype=torch.float32)


def make_loader(data, split, batch_size, seed, shuffle=False):
    return DataLoader(SequenceDataset(data, split), batch_size=batch_size,
                      shuffle=shuffle, num_workers=0, drop_last=False,
                      generator=torch.Generator().manual_seed(seed))


def checked_log1p(x):
    if not torch.isfinite(x).all() or (x < 0).any():
        raise ValueError("Expected finite nonnegative raw counts before log1p.")
    return torch.log1p(x)


def predict_loader(model, loader, device, criterion=None):
    model.eval()
    labels, scores, total_loss, count = [], [], 0.0, 0
    with torch.inference_mode():
        for x, y in loader:
            x = checked_log1p(x.to(device))
            y = y.to(device)
            logits = model(x)
            if not torch.isfinite(logits).all():
                raise ValueError("Nonfinite model predictions.")
            if criterion is not None:
                loss = criterion(logits, y)
                if not torch.isfinite(loss):
                    raise ValueError("Nonfinite validation loss.")
                total_loss += float(loss.item()) * len(y)
            count += len(y)
            labels.append(y.cpu().numpy())
            scores.append(torch.sigmoid(logits).cpu().numpy())
    return total_loss / count, np.concatenate(labels), np.concatenate(scores)


def train_run(data, model_config, settings, run_id):
    if tuple(data["X"].shape[1:]) != (model_config.seq_len, model_config.input_dim):
        raise ValueError("Architecture and tensor dimensions disagree.")
    for name in ("epochs", "patience", "batch_size"):
        if not isinstance(settings[name], int) or settings[name] <= 0:
            raise ValueError(f"{name} must be a positive integer.")
    for name in ("learning_rate", "grad_clip"):
        if not np.isfinite(settings[name]) or settings[name] <= 0:
            raise ValueError(f"{name} must be positive and finite.")
    for name in ("weight_decay", "min_delta"):
        if not np.isfinite(settings[name]) or settings[name] < 0:
            raise ValueError(f"{name} must be nonnegative and finite.")
    seed_everything(settings["seed"])
    device = resolve_device(settings["device"])
    class_counts = {}
    for name in ("train", "validation"):
        y = data["y"][data["indices"][name]]
        positives = int(y.sum())
        negatives = len(y) - positives
        if not positives or not negatives:
            raise ValueError(f"{name} requires both classes.")
        class_counts[name] = {"snapshots": len(y), "positives": positives, "negatives": negatives}
    pos_weight = class_counts["train"]["negatives"] / class_counts["train"]["positives"]
    model = ClaimsTransformer(model_config).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=settings["learning_rate"],
                                 weight_decay=settings["weight_decay"])
    train_loader = make_loader(data, "train", settings["batch_size"], settings["seed"], True)
    validation_loader = make_loader(data, "validation", settings["batch_size"], settings["seed"])
    diagnostic_loader = make_loader(data, "train", settings["batch_size"], settings["seed"])
    best_ap, patience_reference = -np.inf, -np.inf
    without_progress, best_epoch, best_state = 0, None, None
    history = []
    print(f"Training on {device}; TRAIN positive weight = {pos_weight:.4f}.", flush=True)
    for epoch in range(1, settings["epochs"] + 1):
        model.train()
        loss_sum, count = 0.0, 0
        for x, y in train_loader:
            x, y = checked_log1p(x.to(device)), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(x), y)
            if not torch.isfinite(loss):
                raise ValueError("Nonfinite training loss.")
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), settings["grad_clip"], error_if_nonfinite=True)
            optimizer.step()
            loss_sum += float(loss.item()) * len(y)
            count += len(y)
        val_loss, val_y, val_scores = predict_loader(model, validation_loader, device, criterion)
        val_ap = float(average_precision_score(val_y, val_scores))
        train_loss, train_y, train_scores = predict_loader(model, diagnostic_loader, device, criterion)
        train_lift = top10_lift(train_y, train_scores, data["metadata"].iloc[data["indices"]["train"]])
        val_lift = top10_lift(val_y, val_scores, data["metadata"].iloc[data["indices"]["validation"]])
        history.append({"epoch": epoch, "optimization_loss": loss_sum / count,
                        "training_loss": train_loss, "training_top10_lift": train_lift,
                        "validation_top10_lift": val_lift,
                        "training_average_precision": float(average_precision_score(train_y, train_scores)),
                        "validation_loss": val_loss, "validation_average_precision": val_ap,
                        "validation_roc_auc": float(roc_auc_score(val_y, val_scores))})
        if val_ap > best_ap:
            best_ap, best_epoch = val_ap, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if val_ap > patience_reference + settings["min_delta"]:
            patience_reference, without_progress = val_ap, 0
        else:
            without_progress += 1
        print(f"Epoch {epoch:02d}: train loss={loss_sum/count:.5f}; "
              f"validation loss={val_loss:.5f}; validation AP={val_ap:.5f}; "
              f"TRAIN lift={train_lift:.3f}; VALIDATION lift={val_lift:.3f}", flush=True)
        if without_progress >= settings["patience"]:
            print(f"Early stopping; restoring epoch {best_epoch}.", flush=True)
            break
    model.load_state_dict(best_state, strict=True)
    _, val_y, val_scores = predict_loader(model, validation_loader, device, criterion)
    threshold = select_validation_threshold(val_y, val_scores)
    summary = {"run_id": run_id, "training_complete": True,
               "completed_at_utc": datetime.now(timezone.utc).isoformat(),
               "best_epoch": int(best_epoch), "epochs_completed": len(history),
               "best_validation_average_precision": best_ap,
               "validation_threshold": threshold,
               "validation_metrics": classification_metrics(val_y, val_scores, threshold),
               "train_pos_weight": float(pos_weight), "class_counts": class_counts,
               "model_parameter_count": sum(p.numel() for p in model.parameters()),
               "model_config": asdict(model_config), "training_settings": dict(settings),
               "input_hashes": data["hashes"], "resolved_device": str(device),
               "preprocessing": "per-batch log1p of raw counts; no fitted preprocessing",
               "test_inference_performed": False,
               "source_review": "Structural checks only; clinical cutoff, outcome-event exclusion and historical claims availability are not certified by these notebooks.",
               "runtime": runtime_metadata()}
    payload = {"format_version": 1, "training_complete": True, "run_id": run_id,
               "model_state_dict": best_state, "model_config": asdict(model_config),
               "training_settings": dict(settings), "input_hashes": data["hashes"],
               "feature_names": list(data["features"]), "time_steps": list(range(model_config.seq_len)),
               "tensor_shape": [int(v) for v in data["X"].shape],
               "transform": "log1p", "validation_threshold": float(threshold),
               "selection_metric": "validation_average_precision", "best_epoch": int(best_epoch),
               "best_validation_average_precision": best_ap}
    buffer = io.BytesIO()
    torch.save(payload, buffer)
    return buffer.getvalue(), summary, pd.DataFrame(history)


def load_verified_model(blob, data, run_id, device="auto"):
    payload = torch.load(io.BytesIO(blob), map_location="cpu", weights_only=True)
    if (not isinstance(payload, dict) or payload.get("format_version") != 1
            or payload.get("training_complete") is not True or payload.get("run_id") != run_id):
        raise ValueError("Checkpoint is incomplete, unsupported, or belongs to another run.")
    if payload.get("input_hashes") != data["hashes"]:
        raise ValueError("Tensor content, keys, labels, feature order or frozen split changed after training.")
    if (payload.get("feature_names") != data["features"]
            or payload.get("time_steps") != list(range(data["X"].shape[1]))
            or payload.get("tensor_shape") != list(data["X"].shape)):
        raise ValueError("Checkpoint tensor layout differs from current inputs.")
    threshold = payload.get("validation_threshold")
    if threshold is None or not np.isfinite(threshold) or not 0 <= threshold <= 1:
        raise ValueError("Missing valid frozen validation threshold.")
    if payload.get("transform") != "log1p":
        raise ValueError("Unknown preprocessing; evaluation stopped.")
    model = ClaimsTransformer(ModelConfig(**payload["model_config"]))
    model.load_state_dict(payload["model_state_dict"], strict=True)
    resolved = resolve_device(device)
    model.to(resolved).eval()
    return model, payload, resolved

"""Training-only temporal permutation selection and saved-run contracts."""
import copy
from sklearn.model_selection import train_test_split


def release_inputs(data):
    if hasattr(data["X"], "_mmap") and not data["X"]._mmap.closed:
        data["X"]._mmap.close()
    if "temporary_directory" in data:
        data["temporary_directory"].cleanup()


def json_blob(value):
    return canonical_json(value).encode()


def top10_lift(y, scores, metadata):
    y, scores = _validate(y, scores)
    if not 0 < y.sum() < len(y):
        raise ValueError("Lift requires both classes.")
    frame = metadata[["PATIENT_ID", "END_DT"]].reset_index(drop=True).copy()
    if len(frame) != len(y):
        raise ValueError("Unaligned ranking metadata.")
    frame["score"], frame["position"] = scores, np.arange(len(y))
    order = frame.sort_values(["score", "PATIENT_ID", "END_DT"], ascending=[False, True, True]).position.to_numpy()
    k = max(1, int(np.ceil(.1 * len(y))))
    return float(y[order[:k]].mean() / y.mean())


def make_internal_plan(metadata, seed=42):
    """60/20/20 split of TRAIN patients; stopping and ranking patients are separate."""
    train = metadata.loc[metadata.SPLIT.eq("train")]
    patients = train.groupby("PATIENT_ID", sort=True).RESP.max()
    fit_ids, other_ids = train_test_split(patients.index.to_numpy(), test_size=.4,
        stratify=patients.to_numpy(), random_state=seed)
    stop_ids, rank_ids = train_test_split(other_ids, test_size=.5,
        stratify=patients.loc[other_ids].to_numpy(), random_state=seed + 1)
    assignments = {**dict.fromkeys(fit_ids, "fit"), **dict.fromkeys(stop_ids, "stop"),
                   **dict.fromkeys(rank_ids, "rank")}
    plan = train[["PATIENT_ID", "END_DT", "RESP"]].copy()
    plan["ROLE"] = plan.PATIENT_ID.map(assignments)
    # One fixed latest snapshot per ranking patient. Choice does not use labels or scores.
    latest = plan.loc[plan.ROLE.eq("rank")].sort_values(["PATIENT_ID", "END_DT"]).groupby("PATIENT_ID").tail(1)
    counts = latest.groupby("END_DT").PATIENT_ID.transform("size")
    eligible = latest.loc[counts >= 2]
    plan["RANK_EVALUATE"] = plan.index.isin(eligible.index)
    for role in ("fit", "stop", "rank"):
        part = plan.loc[plan.ROLE.eq(role)]
        if set(part.RESP) != {0, 1}:
            raise ValueError("Internal partitions require both classes; inspect TRAIN population.")
    if len(eligible) < 20 or set(eligible.RESP) != {0, 1}:
        raise ValueError("Too few date-matched ranking snapshots with both classes.")
    if len(eligible) < .8 * len(latest):
        raise ValueError("Less than 80% of ranking patients have same-date donors; review ranking design.")
    plan = plan.reset_index(drop=True)
    return plan


def validate_plan(metadata, plan):
    expected = metadata.loc[metadata.SPLIT.eq("train"), ["PATIENT_ID", "END_DT", "RESP"]].reset_index(drop=True)
    if not expected.equals(plan[["PATIENT_ID", "END_DT", "RESP"]].reset_index(drop=True)):
        raise ValueError("Internal plan must contain precisely the original TRAIN snapshots.")
    if set(plan.ROLE) != {"fit", "stop", "rank"} or plan.groupby("PATIENT_ID").ROLE.nunique().max() != 1:
        raise ValueError("Internal roles overlap patients or are invalid.")
    if not plan.RANK_EVALUATE.isin([True, False]).all():
        raise ValueError("Invalid ranking mask.")
    chosen = plan.loc[plan.RANK_EVALUATE]
    if not chosen.ROLE.eq("rank").all() or chosen.PATIENT_ID.duplicated().any():
        raise ValueError("Ranking requires one snapshot per separate ranking patient.")
    if chosen.groupby("END_DT").size().min() < 2:
        raise ValueError("Ranking snapshot has no same-date donor.")
    return plan


def plan_rows(data, plan, role):
    validate_plan(data["metadata"], plan)
    mask = plan.RANK_EVALUATE.to_numpy() if role == "rank" else plan.ROLE.eq(role).to_numpy()
    return np.asarray(data["indices"]["train"])[mask]


class SelectedTensor:
    """Lazy feature slicing, retaining every monthly timestep without a full tensor copy."""
    def __init__(self, source, columns):
        self.source, self.columns = source, np.asarray(columns, dtype=int)
        self.shape = (source.shape[0], source.shape[1], len(columns))

    def __getitem__(self, row):
        if not isinstance(row, (int, np.integer)):
            raise TypeError("Selected tensor uses bounded single-snapshot reads.")
        return self.source[int(row)][:, self.columns]


def selected_view(data, columns, fit=None, stop=None):
    if (not columns or len(set(columns)) != len(columns)
            or any(type(i) is not int or not 0 <= i < len(data["features"]) for i in columns)):
        raise ValueError("Invalid selected feature indices.")
    view = dict(data, X=SelectedTensor(data["X"], columns), features=[data["features"][i] for i in columns])
    if fit is not None:
        view["indices"] = {"train": np.asarray(fit), "validation": np.asarray(stop)}
    return view


def same_date_donors(metadata, seed):
    """A patient derangement within each exact cutoff date; no month shuffling."""
    if metadata.PATIENT_ID.duplicated().any():
        raise ValueError("Use one ranking snapshot per patient.")
    rng = np.random.default_rng(seed)
    donors = np.arange(len(metadata))
    for _, positions in metadata.reset_index(drop=True).groupby("END_DT", sort=True).indices.items():
        if len(positions) < 2:
            raise ValueError("Each cutoff requires at least two patients.")
        order = rng.permutation(positions)
        donors[order] = np.roll(order, 1)
    return donors


def score_rows(model, data, rows, device, batch_size=256, feature=None, donors=None):
    model.eval()
    scores = []
    if feature is not None and (donors is None or len(donors) != len(rows)):
        raise ValueError("Permutation needs one donor per row.")
    with torch.inference_mode():
        for start in range(0, len(rows), batch_size):
            subset = rows[start:start + batch_size]
            values = np.stack([np.asarray(data["X"][int(i)], dtype=np.float32) for i in subset])
            if feature is not None:
                source_rows = rows[donors[start:start + batch_size]]
                values[:, :, feature] = np.stack([data["X"][int(i)][:, feature] for i in source_rows])
            p = torch.sigmoid(model(checked_log1p(torch.as_tensor(values, device=device)))).cpu().numpy()
            scores.append(p)
    scores = np.concatenate(scores)
    _validate(data["y"][rows], scores)
    return scores


def rank_feature(model, data, rows, device, index, repeats, seed, baseline):
    meta, y = data["metadata"].iloc[rows].reset_index(drop=True), data["y"][rows]
    lift_drops, ap_drops = [], []
    for repeat in range(repeats):
        # Common permutations across features reduce unnecessary comparison noise.
        donors = same_date_donors(meta, seed + repeat)
        scores = score_rows(model, data, rows, device, feature=index, donors=donors)
        lift_drops.append(baseline["lift"] - top10_lift(y, scores, meta))
        ap_drops.append(baseline["ap"] - float(average_precision_score(y, scores)))
    return {"feature_index": int(index), "feature_name": data["features"][index],
            "mean_lift_drop": float(np.mean(lift_drops)),
            "std_lift_drop": float(np.std(lift_drops, ddof=1)) if repeats > 1 else 0.,
            "mean_ap_drop": float(np.mean(ap_drops)), "lift_drops": lift_drops}


def selection_from_ranking(records, features, top_k):
    if type(top_k) is not int or not 0 < top_k < len(features):
        raise ValueError("TOP_K must be positive and smaller than the full vocabulary.")
    table = pd.DataFrame(records)
    if (len(table) != len(features) or table.feature_index.duplicated().any()
            or sorted(table.feature_index) != list(range(len(features)))):
        raise ValueError("Ranking is incomplete or has duplicate features.")
    ordered = table.sort_values("feature_index")
    if ordered.feature_name.tolist() != features:
        raise ValueError("Ranking feature names disagree with original vocabulary.")
    if not np.isfinite(table[["mean_lift_drop", "std_lift_drop", "mean_ap_drop"]].to_numpy()).all():
        raise ValueError("Invalid importance values.")
    table = table.sort_values(["mean_lift_drop", "mean_ap_drop", "feature_index"], ascending=[False, False, True]).reset_index(drop=True)
    table["rank"] = np.arange(1, len(table) + 1)
    table["selected"] = table["rank"] <= top_k
    # Model input remains in the original vocabulary order, not importance order.
    columns = sorted(int(i) for i in table.loc[table.selected, "feature_index"])
    return table, columns


def verify_audits(data, preparation, split_audit):
    compare_preparation_to_split(preparation, split_audit)
    if data["hashes"] != preparation["reference_input_hashes"]:
        raise ValueError("Tensor, labels, features or split differs from original RUN_001.")


def model_artifacts(data, config, settings, run_id, contract):
    blob, summary, history = train_run(data, config, settings, run_id)
    summary["contract"] = contract
    return {"checkpoint.pt": blob, "summary.json": json_blob(summary),
            "history.csv": history.to_csv(index=False).encode()}


def checked_model(blobs, data, run_id, contract):
    summary = json.loads(blobs["summary.json"])
    if summary.get("contract") != contract:
        raise ValueError("Saved model uses different code/settings/inputs; choose a new RUN_ID.")
    model, payload, device = load_verified_model(blobs["checkpoint.pt"], data, run_id)
    if payload["model_config"] != summary["model_config"] or payload["training_settings"] != summary["training_settings"]:
        raise ValueError("Checkpoint and summary disagree.")
    return model, payload, device


def plot_history(history, title):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    history.plot(x="epoch", y=["training_loss", "validation_loss"], ax=axes[0], title=title + " loss (eval mode)")
    history.plot(x="epoch", y=["training_top10_lift", "validation_top10_lift"], ax=axes[1], title=title + " top-10% lift")
    fig.tight_layout()
    return fig

def rank_tables(metadata, scores):
    labels = metadata.RESP.to_numpy(dtype=int)
    scores = np.asarray(scores, dtype=float)
    if scores.shape != labels.shape or not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
        raise ValueError("Invalid evaluation probabilities.")
    ranked = metadata[["PATIENT_ID", "END_DT", "RESP"]].copy()
    ranked["SCORE"] = scores
    ranked = ranked.sort_values(["SCORE", "PATIENT_ID", "END_DT"], ascending=[False, True, True]).reset_index(drop=True)
    n, positives = len(ranked), int(labels.sum())
    base = positives / n
    ranked["DECILE"] = 10 - np.minimum(9, np.arange(n) * 10 // n)
    deciles = []
    cumulative_n = cumulative_positive = 0
    for decile in range(10, 0, -1):
        part = ranked[ranked.DECILE == decile]
        if part.empty:
            continue
        count, positive = len(part), int(part.RESP.sum())
        cumulative_n += count
        cumulative_positive += positive
        rate = positive / count
        deciles.append({"decile": decile, "snapshots": count, "positives": positive,
                        "score_min": float(part.SCORE.min()), "score_max": float(part.SCORE.max()),
                        "response_rate": rate, "lift": rate / base if base else None,
                        "cumulative_snapshots": cumulative_n, "cumulative_positives": cumulative_positive,
                        "cumulative_lift": (cumulative_positive / cumulative_n) / base if base else None,
                        "cumulative_recall": cumulative_positive / positives if positives else None})
    top = []
    for fraction in (.05, .10, .20, .30):
        k = max(1, int(np.ceil(n * fraction)))
        tp = int(ranked.RESP.iloc[:k].sum())
        top.append({"fraction": fraction, "selected": k, "positives": tp, "precision": tp / k,
                    "recall": tp / positives if positives else None, "lift": (tp / k) / base if base else None})
    return pd.DataFrame(deciles), pd.DataFrame(top)
IMPLEMENTATION_SHA256 = "c3fdd9194fc16b7e4b1644edb176f75901c1e551cab9eae34d5a8da77b6b2f2d"


In [ ]:
# CELL 3: Load unchanged tensor and verify the declared experiment
if "data" in globals():
    release_inputs(data)
data = load_inputs()
preparation_audit = json.loads(read_artifacts(PREPARATION_TABLE, {"preparation_report.json"})["preparation_report.json"])
split_audit = json.loads(read_artifacts(SPLIT_AUDIT_TABLE, {"split_audit.json"})["split_audit.json"])
verify_audits(data, preparation_audit, split_audit)
plan_info = json.loads(read_artifacts(INTERNAL_TABLE, {"plan.json"})["plan.json"])
if plan_info["seed"] != PLAN_SEED or plan_info["source_hash"] != data["hashes"]["snapshot_manifest_sha256"]:
    raise ValueError("Internal plan differs from this run.")
plan = validate_plan(data["metadata"], pd.DataFrame(plan_info["records"]))
if plan.to_dict("records") != make_internal_plan(data["metadata"], PLAN_SEED).to_dict("records"):
    raise ValueError("Internal plan does not reproduce the declared patient split.")

if type(PERMUTATION_REPEATS) is not int or PERMUTATION_REPEATS < 2:
    raise ValueError("Use at least two permutation repetitions.")
if type(FEATURES_PER_CHUNK) is not int or FEATURES_PER_CHUNK < 1:
    raise ValueError("FEATURES_PER_CHUNK must be positive.")
if type(TOP_K) is not int or not 0 < TOP_K < len(data["features"]):
    raise ValueError("TOP_K must be between 1 and the original feature count minus one.")
reference = json.loads(read_artifacts(REFERENCE_MODEL_TABLE, REFERENCE_TRAINING_ARTIFACT_NAMES)["training_summary.json"])
if any(reference["model_config"].get(k) != v for k, v in MODEL_SETTINGS.items()):
    raise ValueError("Architecture differs from original RUN_001; inspect original settings.")
if any(reference["training_settings"].get(k) != v for k, v in TRAINING_SETTINGS.items() if k != "device"):
    raise ValueError("Training settings differ from original RUN_001; inspect original settings.")
contract = {"implementation": IMPLEMENTATION_SHA256, "input_hashes": data["hashes"],
            "plan_sha256": digest_json(plan_info), "top_k": TOP_K,
            "repeats": PERMUTATION_REPEATS, "permutation_seed": PERMUTATION_SEED,
            "chunk_size": FEATURES_PER_CHUNK, "model": MODEL_SETTINGS, "training": TRAINING_SETTINGS}
fit_rows, stop_rows, ranking_rows = [plan_rows(data, plan, role) for role in ("fit", "stop", "rank")]
all_columns = list(range(len(data["features"])))
ranking_data = selected_view(data, all_columns, fit_rows, stop_rows)
print("Original tensor:", data["X"].shape, "| Final selected tensor:", (len(data["X"]), 12, TOP_K))
print("Importance scoring needs", len(all_columns) * PERMUTATION_REPEATS, "holdout inference passes; GPU recommended.")


In [ ]:
# CELL 4: Fit the ranking encoder on internal fitting patients only
model_names = {"checkpoint.pt", "summary.json", "history.csv"}
if not table_exists(RANK_MODEL_TABLE):
    artifacts = model_artifacts(ranking_data, ModelConfig(input_dim=len(all_columns), **MODEL_SETTINGS),
        TRAINING_SETTINGS, RUN_ID + "_RANK", contract)
    save_artifacts(RANK_MODEL_TABLE, artifacts)
rank_blobs = read_artifacts(RANK_MODEL_TABLE, model_names)
rank_model, rank_payload, rank_device = checked_model(rank_blobs, ranking_data, RUN_ID + "_RANK", contract)
print("Ranking encoder checkpoint chosen using internal stopping patients only.")


In [ ]:
# CELL 5: Rank all original features by same-date whole-sequence permutation
ranking_meta = data["metadata"].iloc[ranking_rows].reset_index(drop=True)
ranking_y = data["y"][ranking_rows]
unshuffled = score_rows(rank_model, data, ranking_rows, rank_device)
baseline = {"lift": top10_lift(ranking_y, unshuffled, ranking_meta),
            "ap": float(average_precision_score(ranking_y, unshuffled))}
print("Independent internal ranking baseline:", baseline)
if baseline["lift"] <= 1:
    raise ValueError("Ranking encoder has no top-decile enrichment on its independent holdout; do not trust this ranking.")
records = []
rank_contract = {"contract": contract, "checkpoint_sha256": hashlib.sha256(rank_blobs["checkpoint.pt"]).hexdigest()}
for start in range(0, len(all_columns), FEATURES_PER_CHUNK):
    end = min(start + FEATURES_PER_CHUNK, len(all_columns))
    table = RUN_PREFIX + f"_IMPORTANCE_{start:04d}"
    if table_exists(table):
        chunk = json.loads(read_artifacts(table, {"importance.json"})["importance.json"])
        if (chunk["contract"] != rank_contract or chunk["start"] != start or chunk["end"] != end
                or not np.isclose(chunk["baseline"]["lift"], baseline["lift"], rtol=0, atol=1e-10)
                or not np.isclose(chunk["baseline"]["ap"], baseline["ap"], rtol=0, atol=1e-8)):
            raise ValueError("Saved ranking chunk differs; use the matching runtime or a new RUN_ID.")
    else:
        items = []
        for feature_index in range(start, end):
            result = rank_feature(rank_model, data, ranking_rows, rank_device, feature_index,
                                  PERMUTATION_REPEATS, PERMUTATION_SEED, baseline)
            items.append(result)
            print(f"Feature {feature_index+1}/{len(all_columns)}: lift drop={result['mean_lift_drop']:.4f}", flush=True)
        chunk = {"contract": rank_contract, "start": start, "end": end, "baseline": baseline, "records": items}
        save_artifacts(table, {"importance.json": json_blob(chunk)})
    if [r["feature_index"] for r in chunk["records"]] != list(range(start, end)):
        raise ValueError("Ranking chunk indices are incomplete or reordered.")
    records.extend(chunk["records"])
ranking, selected_indices = selection_from_ranking(records, data["features"], TOP_K)
manifest = {"contract": contract, "selected_indices": selected_indices,
            "selected_features": [data["features"][i] for i in selected_indices],
            "source_features": data["features"], "ranking_baseline": baseline,
            "ranking_patients": len(ranking_rows), "ranking_positives": int(ranking_y.sum()),
            "ranking_checkpoint_sha256": rank_contract["checkpoint_sha256"],
            "selection_rule": "mean_top10_lift_drop_then_mean_AP_drop_then_original_feature_index",
            "time_steps": list(range(12)), "test_used": False, "outer_validation_used_for_ranking": False}
save_artifacts(SELECTION_TABLE, {"manifest.json": json_blob(manifest), "ranking.json": json_blob(records),
                               "ranking.csv": ranking.to_csv(index=False).encode()})
display(ranking.drop(columns="lift_drops"))
print("Selected features with nonpositive measured lift drop:", int((ranking.loc[ranking.selected, "mean_lift_drop"] <= 0).sum()))
print("Top 50 is a fixed experiment, not proof that every selected feature is relevant. Correlated features can mask importance.")
del rank_model, rank_payload, rank_blobs
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# CELL 6: Retrain the original encoder on full TRAIN with selected monthly features
selected_data = selected_view(data, selected_indices)
final_contract = {"experiment": contract, "selection_sha256": digest_json(manifest)}
if not table_exists(MODEL_TABLE):
    artifacts = model_artifacts(selected_data, ModelConfig(input_dim=len(selected_indices), **MODEL_SETTINGS),
                                TRAINING_SETTINGS, RUN_ID, final_contract)
    save_artifacts(MODEL_TABLE, artifacts)
final_blobs = read_artifacts(MODEL_TABLE, model_names)
model, payload, device = checked_model(final_blobs, selected_data, RUN_ID, final_contract)
summary = json.loads(final_blobs["summary.json"])
print("Selected input shape:", selected_data["X"].shape, "| Best epoch:", summary["best_epoch"])


In [ ]:
# CELL 7: Training and validation curves, lift and deciles
import matplotlib.pyplot as plt
history = pd.read_csv(io.BytesIO(final_blobs["history.csv"]))
plot_history(history, "Selected monthly features")
plt.show()
for split in ("train", "validation"):
    _, labels, scores = predict_loader(model, make_loader(selected_data, split, TRAINING_SETTINGS["batch_size"], 42), device)
    meta = data["metadata"].iloc[data["indices"][split]]
    deciles, top = rank_tables(meta, scores)
    print(split.upper(), "top-10% lift:", top10_lift(labels, scores, meta))
    display(deciles)
    display(top)


In [ ]:
# CELL 8: Training complete; retain the frozen feature selection for evaluation
print("Run notebook 04 with the same RUN_ID. No TEST scores were used for ranking or training.")
print("All 12 monthly timesteps are retained; only the feature axis was reduced.")
release_inputs(data)
del model, selected_data, ranking_data, data
